# 03 — Model Training

Trains two XGBoost models on the feature matrix produced by
`02_feature_engineering.ipynb`:

| Model       | Task                                 | Output path                        |
|-------------|--------------------------------------|------------------------------------|
| Classifier  | Will the invoice be delayed? (0/1)   | `ml_models/classifier/model.joblib`|
| Regressor   | How many days late?                  | `ml_models/regressor/model.joblib` |

The saved paths match what `backend/app/ml/model_loader.py` expects, so once
these files exist the API will automatically use them for inference.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)
from xgboost import XGBClassifier, XGBRegressor

sns.set_theme(style="whitegrid", palette="viridis", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 120

PROJECT_ROOT = Path.cwd().parent
DATA_DIR     = PROJECT_ROOT / "data"
MODELS_DIR   = PROJECT_ROOT / "ml_models"

RANDOM_STATE = 42
print("Libraries loaded ✓")

## 1 · Load feature matrix

In [ ]:
features_path = DATA_DIR / "features.parquet"
assert features_path.exists(), f"Feature file not found: {features_path}. Run 02_feature_engineering.ipynb first."

df = pd.read_parquet(features_path)
print(f"Loaded feature matrix: {df.shape}")
df.head(3)

In [ ]:
# Feature columns — must match backend/app/ml/feature_engineering.FEATURE_COLUMNS
FEATURE_COLUMNS = [
    "invoice_amount",
    "days_until_due",
    "invoice_age",
    "payment_term_net_days",
    "is_recurring",
    "avg_payment_days",
    "late_payment_ratio",
    "credit_limit",
    "customer_tenure_days",
    "amount_to_credit_ratio",
    "month_issued",
    "day_of_week_issued",
    "quarter_issued",
    "is_month_end",
    "is_quarter_end",
]

X = df[FEATURE_COLUMNS].copy()
y_cls = df["is_delayed"].copy()           # classification target
y_reg = df["delay_days"].copy().astype(float)  # regression target

print(f"Features : {X.shape}")
print(f"Class balance:\n{y_cls.value_counts(normalize=True).to_string()}")

## 2 · Train / test split

In [ ]:
X_train, X_test, y_cls_train, y_cls_test, y_reg_train, y_reg_test = train_test_split(
    X, y_cls, y_reg,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_cls,
)
print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")
print(f"Train delay rate: {y_cls_train.mean():.2%}  |  Test delay rate: {y_cls_test.mean():.2%}")

---
## 3 · XGBoost Classifier — will the invoice be late?

In [ ]:
# Handle class imbalance via scale_pos_weight
n_neg = (y_cls_train == 0).sum()
n_pos = (y_cls_train == 1).sum()
scale_pos_weight = n_neg / max(n_pos, 1)
print(f"scale_pos_weight = {scale_pos_weight:.2f}  (neg={n_neg:,}, pos={n_pos:,})")

clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    n_jobs=-1,
)

clf.fit(
    X_train, y_cls_train,
    eval_set=[(X_train, y_cls_train), (X_test, y_cls_test)],
    verbose=50,
)

print("\n✓ Classifier training complete")

### 3.1 · Classifier — cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(clf, X_train, y_cls_train, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"5-fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Per-fold scores  : {np.round(cv_scores, 4)}")

### 3.2 · Classifier — test-set evaluation

In [ ]:
y_cls_pred  = clf.predict(X_test)
y_cls_proba = clf.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_cls_test, y_cls_pred)
f1   = f1_score(y_cls_test, y_cls_pred)
auc  = roc_auc_score(y_cls_test, y_cls_proba)

print(f"Test Accuracy : {acc:.4f}")
print(f"Test F1       : {f1:.4f}")
print(f"Test ROC-AUC  : {auc:.4f}")
print()
print(classification_report(y_cls_test, y_cls_pred, target_names=["On-Time", "Delayed"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 3.2a — Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_cls_test, y_cls_pred,
    display_labels=["On-Time", "Delayed"],
    cmap="Blues", ax=axes[0],
)
axes[0].set_title("Confusion Matrix")

# 3.2b — ROC curve
fpr, tpr, _ = roc_curve(y_cls_test, y_cls_proba)
axes[1].plot(fpr, tpr, color="#e74c3c", lw=2, label=f"AUC = {auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend(loc="lower right")

# 3.2c — Precision-Recall curve
precision, recall, _ = precision_recall_curve(y_cls_test, y_cls_proba)
ap = average_precision_score(y_cls_test, y_cls_proba)
axes[2].plot(recall, precision, color="#3498db", lw=2, label=f"AP = {ap:.4f}")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall Curve")
axes[2].legend(loc="lower left")

plt.suptitle("Classifier Evaluation", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.3 · Classifier — feature importance

In [ ]:
clf_importances = pd.Series(
    clf.feature_importances_, index=FEATURE_COLUMNS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
clf_importances.plot.barh(ax=ax, color="#e74c3c", edgecolor="black")
ax.set_title("XGBoost Classifier — Feature Importance (gain)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

---
## 4 · XGBoost Regressor — how many days late?

In [ ]:
# Train only on delayed invoices for the regressor
delayed_mask_train = y_cls_train == 1
delayed_mask_test  = y_cls_test == 1

X_reg_train = X_train[delayed_mask_train]
y_reg_train_d = y_reg_train[delayed_mask_train]
X_reg_test  = X_test[delayed_mask_test]
y_reg_test_d = y_reg_test[delayed_mask_test]

print(f"Regressor train: {X_reg_train.shape[0]:,}  |  test: {X_reg_test.shape[0]:,}")

reg = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

reg.fit(
    X_reg_train, y_reg_train_d,
    eval_set=[(X_reg_train, y_reg_train_d), (X_reg_test, y_reg_test_d)],
    verbose=50,
)

print("\n✓ Regressor training complete")

### 4.1 · Regressor — test-set evaluation

In [ ]:
y_reg_pred = reg.predict(X_reg_test)
y_reg_pred = np.clip(np.round(y_reg_pred), 0, None)  # clamp to >= 0

mae  = mean_absolute_error(y_reg_test_d, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test_d, y_reg_pred))
r2   = r2_score(y_reg_test_d, y_reg_pred)

print(f"Test MAE  : {mae:.2f} days")
print(f"Test RMSE : {rmse:.2f} days")
print(f"Test R²   : {r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 4.1a — Actual vs predicted
axes[0].scatter(y_reg_test_d, y_reg_pred, alpha=0.4, s=20, color="#3498db")
max_val = max(y_reg_test_d.max(), y_reg_pred.max())
axes[0].plot([0, max_val], [0, max_val], "r--", lw=2, label="Perfect")
axes[0].set_xlabel("Actual delay days")
axes[0].set_ylabel("Predicted delay days")
axes[0].set_title("Actual vs Predicted Delay Days")
axes[0].legend()

# 4.1b — Residuals
residuals = y_reg_test_d.values - y_reg_pred
axes[1].hist(residuals, bins=40, color="#9b59b6", edgecolor="black", alpha=0.8)
axes[1].axvline(0, color="red", ls="--", lw=2)
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual (actual − predicted)")

# 4.1c — Residuals vs predicted
axes[2].scatter(y_reg_pred, residuals, alpha=0.4, s=20, color="#e67e22")
axes[2].axhline(0, color="red", ls="--", lw=2)
axes[2].set_xlabel("Predicted delay days")
axes[2].set_ylabel("Residual")
axes[2].set_title("Residuals vs Predicted")

plt.suptitle("Regressor Evaluation", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4.2 · Regressor — feature importance

In [ ]:
reg_importances = pd.Series(
    reg.feature_importances_, index=FEATURE_COLUMNS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
reg_importances.plot.barh(ax=ax, color="#3498db", edgecolor="black")
ax.set_title("XGBoost Regressor — Feature Importance (gain)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

---
## 5 · Combined evaluation — full pipeline

In [ ]:
# Simulate the prediction pipeline: classifier first, then regressor for positives
y_pipe_cls   = clf.predict(X_test)
y_pipe_proba = clf.predict_proba(X_test)[:, 1]

y_pipe_days = np.zeros(len(X_test))
delayed_idx = y_pipe_cls == 1
if delayed_idx.sum() > 0:
    y_pipe_days[delayed_idx] = np.clip(
        np.round(reg.predict(X_test[delayed_idx])), 0, None
    )

print("=" * 60)
print("FULL PIPELINE EVALUATION (test set)")
print("=" * 60)
print(f"Classifier accuracy  : {accuracy_score(y_cls_test, y_pipe_cls):.4f}")
print(f"Classifier F1        : {f1_score(y_cls_test, y_pipe_cls):.4f}")
print(f"Classifier ROC-AUC   : {roc_auc_score(y_cls_test, y_pipe_proba):.4f}")

# Regressor metrics only for truly delayed invoices that were correctly classified
correct_delayed = (y_pipe_cls == 1) & (y_cls_test.values == 1)
if correct_delayed.sum() > 0:
    pipe_mae = mean_absolute_error(
        y_reg_test[correct_delayed].values,
        y_pipe_days[correct_delayed],
    )
    print(f"Regressor MAE (TP)   : {pipe_mae:.2f} days")
print("=" * 60)

---
## 6 · Save models with joblib

Saved to `ml_models/classifier/model.joblib` and `ml_models/regressor/model.joblib`
which is exactly where `backend/app/ml/model_loader.py` loads from.

In [ ]:
clf_dir = MODELS_DIR / "classifier"
reg_dir = MODELS_DIR / "regressor"
clf_dir.mkdir(parents=True, exist_ok=True)
reg_dir.mkdir(parents=True, exist_ok=True)

clf_path = clf_dir / "model.joblib"
reg_path = reg_dir / "model.joblib"

joblib.dump(clf, clf_path)
joblib.dump(reg, reg_path)

print(f"✓ Classifier saved → {clf_path}  ({clf_path.stat().st_size / 1024:.1f} KB)")
print(f"✓ Regressor  saved → {reg_path}  ({reg_path.stat().st_size / 1024:.1f} KB)")

## 7 · Verify saved models re-load correctly

In [ ]:
clf_loaded = joblib.load(clf_path)
reg_loaded = joblib.load(reg_path)

# Quick sanity check — predictions should be identical
assert np.allclose(
    clf.predict_proba(X_test[:5]),
    clf_loaded.predict_proba(X_test[:5]),
), "Classifier predictions mismatch after reload!"

assert np.allclose(
    reg.predict(X_reg_test[:5]),
    reg_loaded.predict(X_reg_test[:5]),
), "Regressor predictions mismatch after reload!"

print("✓ Models reload and produce identical predictions")

## 8 · Summary metrics for model registry

In [ ]:
import json

metrics = {
    "classifier": {
        "accuracy": round(acc, 4),
        "f1_score": round(f1, 4),
        "roc_auc": round(auc, 4),
        "cv_roc_auc_mean": round(cv_scores.mean(), 4),
        "cv_roc_auc_std": round(cv_scores.std(), 4),
    },
    "regressor": {
        "mae": round(mae, 2),
        "rmse": round(rmse, 2),
        "r2": round(r2, 4),
    },
    "training_info": {
        "n_train": int(X_train.shape[0]),
        "n_test": int(X_test.shape[0]),
        "delay_rate_train": round(float(y_cls_train.mean()), 4),
        "delay_rate_test": round(float(y_cls_test.mean()), 4),
        "features": FEATURE_COLUMNS,
        "random_state": RANDOM_STATE,
    },
}

metrics_path = MODELS_DIR / "training_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"✓ Metrics saved → {metrics_path}")
print(json.dumps(metrics, indent=2))

In [ ]:
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Classifier : {clf_path}")
print(f"Regressor  : {reg_path}")
print(f"Metrics    : {metrics_path}")
print("\nThe API server will now automatically use these models")
print("(restart the server or call the /reload endpoint).")
print("=" * 60)